[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/koebe/koebe.ipynb)

# Koebe Polyhedra

This example computes the Koebe polyhedron of a triangulation of the sphere, realizing its incidence pattern as a convex polyhedron whose edges all touch the unit sphere. Its polar dual touches the sphere at the very same points, with each dual edge crossing its primal edge there at right angles so every dual vertex lies inside its face and every dual edge has positive length. By the Koebe-Andreev-Thurston circle-packing theorem, the tangency conditions determine the polyhedron up to the sphere-preserving transformations, which first centre the tangency points and then sweep the polyhedron through a family of shapes.

Space is modelled with the unit sphere as its metric, where spatial planes square to one and the weight plane squares to minus one. Each circle cut out on the sphere is a unit vector `Plane`, and each point is an antivector `Point` that lies on the sphere when it is null and on a circle when `circle & point == 0`. Dualizing a circle `circle.dual()` gives the apex of the cone tangent to the sphere along that circle, two circles touch externally when `circle | other == -1` at the null point `circle.dual() + other.dual()`, and the dual `(a ^ b ^ c).dual()` of three vertex circles around a face is the orthogonal face circle. Because rotors preserve every inner product, sandwiching `rotor >> circles` carries any touching pattern to another touching pattern.

In stereographic and Möbius-transformation notation, circles on $S^2$ project to planar circles or lines via $(u, v) = (x/(1 - z), y/(1 - z))$, external tangency writes Euclidean centre distances as $\|\mathbf c_i - \mathbf c_j\| = r_i + r_j$ in the plane or $\mathbf n_i \cdot \mathbf n_j = -\cos(\rho_i + \rho_j)$ in spherical cap radii $\rho_i$ around unit normals $\mathbf n_i \in S^2$, cone apexes stand at $\mathbf n_i / \cos\rho_i$, and sphere-preserving transformations act by fractional linear maps $w \mapsto (a w + b)/(c w + d)$ with $ad - bc = 1$ between forward and inverse stereographic projections.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import Sparse, spdiag
from numga.topology import Topology
from examples.util.animation import save_figure, show_states
from examples.surfaces.koebe import render, scenarios

ga = Algebra("x+y+z+w-")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                 # Scalar+
# Extrinsic planes cutting the unit sphere along oriented circles.
Plane = ga.gatype.vector()                                                  # Vector+
# Intrinsic face circles dual to three wedged vertex planes.
Patch = Plane.derive.flipped_orientation                                    # Vector-
# Points of space and null points of the sphere, dual to polar planes.
Point = ga.gatype.antivector()                                              # Antivector-
Rotor = ga.gatype.rotor()                                                   # Rotor+
# Centre of the unit sphere, dual to the weight plane w.
origin = mv.w.dual()                                                        # [] Point-

## Circles touching along every edge

Starting from an irregular convex-hull triangulation of the sphere, spherical caps of uniform angular radius around the vertex directions initialize the circle planes. Gauss-Newton iterations then drive every vertex plane to unit square and every edge pair to inner product minus one. Because minimum-norm steps in the circle metric commute with the algebra's six-dimensional rotor group, the solver leaves those six neutral directions undisturbed.

In [ ]:
topology, direction_coords = scenarios.sphere_triangulation()
directions = (stack((mv.x, mv.y, mv.z)) * direction_coords).sum(axis=-1).dual().field()  # Point-[V]
n_newton_steps = scenarios.n_newton_steps

In [ ]:
def cap_circles(topology: Topology, directions: Point) -> Plane:
    """Spherical caps around `directions` (`Point-[V]`) of radius half the mean edge angle initialize the circle planes (`Plane+[V]`)."""
    tails, heads = topology.incidence[1, 0] * directions                    # Point-[E], Point-[E]
    angles = (heads * tails).log().norm()                                   # Scalar+[E]
    radius = angles.sites.mean() / 2                                        # [] Scalar+
    # Place each cone apex along its direction at reciprocal cosine distance so its dual circle plane squares to one.
    apexes = (directions + origin * radius.cos()) / radius.sin()            # Point-[V]
    return apexes.dual()                                                    # Plane+[V]


def koebe_circles(topology: Topology, start: Plane, n_steps: int) -> Plane:
    """Gauss-Newton iterations from `start` (`Plane+[V]`) solve for unit circles touching externally along every edge (`Plane+[V]`)."""
    V, E, F = topology.n_elements
    vertex_rows = Sparse.selection(np.arange(V), V + E).adjugate()          # Scalar+[V + E <- V]
    edge_rows = Sparse.selection(V + np.arange(E), V + E).adjugate()        # Scalar+[V + E <- E]
    circles = start                                                         # Plane+[V]
    for _ in range(n_steps):
        tails, heads = topology.incidence[1, 0] * circles                   # Plane+[E], Plane+[E]
        residual = vertex_rows * (circles.squared() - 1) + edge_rows * ((tails | heads) + 1)  # Scalar+[V + E]
        # Linearize unit-square residuals on vertices and tangency inner products across incident edge endpoints.
        vertex_part = spdiag(2 * circles | Plane)                           # (Scalar+ <- Plane+)[V <- V]
        edge_part = (spdiag(stack([heads, tails]) | Plane) * topology.incidence[1, 0]).sum(axis=0)  # (Scalar+ <- Plane+)[E <- V]
        jacobian = vertex_rows * vertex_part + edge_rows * edge_part        # (Scalar+ <- Plane+)[V + E <- V]
        # Minimum-norm steps in the circle metric commute with sphere-preserving rotors and leave their six null directions untouched.
        circles = circles - jacobian.adjoint()(jacobian(jacobian.adjoint()).solve(residual))
    return circles


start = cap_circles(topology, directions)                                   # Plane+[V]
circles = koebe_circles(topology, start, n_newton_steps)                    # Plane+[V]
start_tails, start_heads = topology.incidence[1, 0] * start                 # Plane+[E], Plane+[E]
start_touch = start_tails | start_heads                                     # Scalar+[E]
print(f"initial edge inner products span {start_touch.to_array().min():.2f} to {start_touch.to_array().max():.2f}")

## Polar dual polyhedron and centring

Summing the poles at the two ends of each edge gives its null tangency point on the sphere, while dualizing the wedge of three vertex circles around each face gives the orthogonal face circle whose cone apex is a vertex of the polar dual polyhedron. Minimizing the summed logarithms of minus the inner products with the tangency points locates their unique timelike balance point inside the sphere, and the boost carrying that point onto `origin` centres the polyhedron.

In [ ]:
n_centre_iterations = scenarios.n_centre_iterations

In [ ]:
def tangency_points(topology: Topology, circles: Plane) -> Point:
    """Summing the poles at each edge's endpoints locates the null tangency points on the sphere (`[...] Point-[E]`)."""
    return topology.incidence[1, 0].sum(axis=0) * circles.dual()            # [...] Point-[E]


def cone_apexes(circles: Plane | Patch) -> Point:
    """Dualizing `circles` and normalizing by `w` places the tangent-cone apexes at unit weight (`[...] Point-`)."""
    poles = circles.dual()                                                  # [...] Antivector+-
    return poles / (mv.w & poles)                                           # [...] Point-

In [ ]:
def face_circles(topology: Topology, circles: Plane) -> Patch:
    """Dualizing the wedge of each triangle's three vertex circles gives the orthogonal face circles (`[...] Patch-[F]`)."""
    corners = topology.incidence[2, 0] * circles[..., None]                 # [..., 3] Plane+[F]
    return (corners[..., 0] ^ corners[..., 1] ^ corners[..., 2]).dual()     # [...] Patch-[F]


def balance_centre(points: Point, n_iterations: int) -> Point:
    """Fixed-point iteration inside the sphere finds the unit-timelike point (`[] Point-`) minimizing summed log inner products with `points`."""
    # At the stationary point, weighting each null point by minus the reciprocal of its inner product sums parallel to the centre.
    centre = origin                                                         # [] Point-
    for _ in range(n_iterations):
        weighted_sum = (points * (-1 / (centre | points))).sites.sum()      # [] Point-
        centre = weighted_sum.normalized()
    return centre


centre = balance_centre(tangency_points(topology, circles), n_centre_iterations)  # [] Point-
centring = (1 - origin * centre).normalized()                               # [] Rotor+
centred_circles = centring >> circles                                       # Plane+[V]
save_figure(render.polyhedron_figure(topology, centred_circles), "koebe");

On the left, blue primal edges and orange dual edges cross at right angles at the dark tangency points on the opaque unit sphere; on the right, the blue vertex circles touch externally while the orange face circles pass orthogonally through each triangle's three tangencies.

## Sphere-preserving boosts

Rotors generated by bivectors through the timelike plane crowd points of the sphere toward one direction while expanding the opposite side. Because rotors preserve all inner products, every boosted image remains a tangent Koebe polyhedron paired with its orthogonal dual.

In [ ]:
rapidity = scenarios.rapidity
tilt = scenarios.tilt
n_frames = scenarios.n_frames
duration_ms = scenarios.duration_ms
boost_view = render.polyhedron_view(scenarios.boost_bounds(centred_circles), render.n_animation_pixels)

In [ ]:
def boost_rotors(rapidity: float, tilt: float, n_frames: int) -> Rotor:
    """Boost rotors (`[n_frames] Rotor+`) sweep a plane through `w` tilted by `tilt` once around the `z`-axis up to `rapidity`."""
    turn_angles = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)       # [n_frames]
    tilted_plane = (mv.zx * (tilt / 2)).exp() >> mv.xw                      # [] Bivector+
    boost_planes = (mv.xy * (-turn_angles / 2)).exp() >> tilted_plane       # [n_frames] Bivector+
    return (boost_planes * (rapidity * np.sin(turn_angles) ** 2 / 2)).exp()  # [n_frames] Rotor+


def boost_states(circles: Plane, rapidity: float, tilt: float, n_frames: int) -> Iterator[Plane]:
    """Yields `circles` (`Plane+[V]`) carried across `n_frames` by `boost_rotors(rapidity, tilt, n_frames)`."""
    yield from boost_rotors(rapidity, tilt, n_frames) >> circles            # [n_frames] Plane+[V]


recorded_boosts = show_states(
    boost_states(centred_circles, rapidity, tilt, n_frames),
    render.boost_animation, topology, boost_view,
    name="koebe_boost", duration_ms=duration_ms, n_frames=n_frames,
);

As the tilted boost plane turns around the vertical axis, vertex and face circles shrink on the crowded side of the sphere and expand opposite while every primal and dual edge stays tangent to the sphere.

In [ ]:
# checks
def line_distances(incidence: Sparse, apexes: Point) -> Scalar:
    """Euclidean distances from `origin` to the lines joining incident endpoint pairs in `apexes` (`[...] Scalar+[E]`)."""
    places = incidence * apexes[..., None]                                  # [..., 2] Point-[E]
    lines = places[..., 0] & places[..., 1]                                 # [...] Antibivector-[E]
    return ((lines | mv.w).squared() / (lines ^ mv.w).squared()).square_root()


# Solved circles square to one and touch externally along every edge.
assert ((circles.squared() - 1).abs() < 1e-10).all()
solved_ends = topology.incidence[1, 0] * circles                            # [2] Plane+[E]
assert (((solved_ends[0] | solved_ends[1]) + 1).abs() < 1e-10).all()

# Boosted circles keep unit square, external tangency, face orthogonality, and unit edge-to-origin distance.
boosted_circles = stack(recorded_boosts)                                    # [n_frames] Plane+[V]
assert ((boosted_circles.squared() - 1).abs() < 1e-10).all()
boosted_ends = topology.incidence[1, 0] * boosted_circles[..., None]        # [n_frames, 2] Plane+[E]
assert (((boosted_ends[..., 0] | boosted_ends[..., 1]) + 1).abs() < 1e-10).all()
duals = face_circles(topology, boosted_circles)                             # [n_frames] Patch-[F]
corners = topology.incidence[2, 0] * boosted_circles[..., None]             # [n_frames, 3] Plane+[F]
assert (((duals[..., None] | corners).squared() / duals.squared()[..., None]).abs() < 1e-20).all()
assert ((line_distances(topology.incidence[1, 0], cone_apexes(boosted_circles)) - 1).abs() < 1e-10).all()
assert ((line_distances(topology.dual.incidence[1, 0], cone_apexes(duals)) - 1).abs() < 1e-10).all()

# Centring places the unit-weight mean of the tangency points at the centre of the sphere.
centred_touches = tangency_points(topology, centred_circles)                # Point-[E]
balance = (centred_touches / (mv.w & centred_touches)).sites.mean()         # [] Point-
assert ((stack((mv.x, mv.y, mv.z)) & balance).abs() < 1e-4).all()